# El problema de las N reinas

Coloca una reina por columna de modo que ninguna pareja comparta fila o diagonal. El estado `(1, 3, 0, 2)` indica la fila de cada reina, de izquierda a derecha. Primero colocaremos las reinas una a una. Después empezaremos con todas colocadas e intentaremos reducir los conflictos.

Compara estos dos tableros de cuatro reinas. En el primero, `(0, 1, 2, 3)`, todas están en la misma diagonal. El segundo, `(1, 3, 0, 2)`, es válido.

```text
Q...    ..Q.
.Q..    Q...
..Q.    ...Q
...Q    .Q..
```

In [ ]:
from random import Random

def xoquen(a, fila_a, b, fila_b):
    return fila_a == fila_b or abs(a - b) == abs(fila_a - fila_b)

def mostra_tauler(estat):
    for fila in range(len(estat)):
        print(" ".join("♛" if f == fila else "·" for f in estat))

## Backtracking
Probamos una fila para la siguiente columna. Si entra en conflicto con alguna reina ya colocada, la descartamos. Si ninguna fila lleva a una solución, volvemos a la columna anterior.

In [ ]:
def backtracking(n, estat=()):
    columna = len(estat)
    if columna == n:
        return estat

    for fila in range(n):
        if any(xoquen(i, f, columna, fila) for i, f in enumerate(estat)):
            continue
        solucio = backtracking(n, estat + (fila,))
        if solucio is not None:
            return solucio
    return None

mostra_tauler(backtracking(8))

## Min-conflicts

A partir de 18 reinas, el tiempo de ejecución es muy alto porque el espacio de búsqueda crece rápidamente. Se conocen 666.090.624 disposiciones válidas para 18 reinas y 2.691.008.701.644 para 22 ([secuencia A000170 de OEIS](https://oeis.org/A000170)). Estas cifras cuentan soluciones válidas; no indican cuántos estados tendrá que visitar el algoritmo de backtracking antes de encontrar una.

Empezamos con una fila aleatoria por columna. En cada paso elegimos una reina en conflicto y la ponemos en una de las filas que menos conflictos le causa. Si hay empate, elegimos al azar. Si agotamos el límite de pasos, podemos probar otras semillas.

In [ ]:
def conflictes(estat, columna, fila):
    return sum(xoquen(i, f, columna, fila)
               for i, f in enumerate(estat) if i != columna)

def min_conflicts(n, rng, passos=1000):
    estat = [rng.randrange(n) for _ in range(n)]
    for pas in range(passos + 1):
        problematiques = [c for c, f in enumerate(estat) if conflictes(estat, c, f)]
        if not problematiques:
            return tuple(estat)
        if pas == passos:
            return None
        columna = rng.choice(problematiques)
        costos = [conflictes(estat, columna, fila) for fila in range(n)]
        millors_files = [fila for fila, cost in enumerate(costos) if cost == min(costos)]
        estat[columna] = rng.choice(millors_files)

solucio = min_conflicts(8, Random(42))
if solucio is not None:
    mostra_tauler(solucio)
else:
    print("Sense solució / Sin solución")